# PolicyProof AI — Real paired base versus trained adapter (48-case development set)

**Read first:** This notebook runs actual GPU inference, **not training**. The 48 cases were inspected during development: this is a diagnostic, not a blind evaluation. No independent human semantic validation is asserted. The existing 60-case proposed final candidate and the 72 unscored challenges are NOT used.

Your previously uploaded `policyproof_t4_torch_extended.zip` has structurally valid LoRA weights and a saved 40-step receipt. The *exact loaded model revision during training* could not be independently inferred, even though the Hugging Face revision was recorded. This notebook loads the same **specified** base commit af89d219b53ed9b13b8a4645f9c8028973510324 anew for both inference passes.

**A T4/P100 GPU, access to gated Tiny Aya Global, and your original uploaded ZIP are required.** Training is OFF and GPU inference is OFF by default. First run the preflight cells; explicitly opt into the expensive GPU step only after they pass.

## 1. Connect GPU and check out latest source. Does not load models or train.

In [ ]:
import os, sys, subprocess
from pathlib import Path
import torch
assert torch.cuda.is_available(), "Choose Runtime > Change runtime type > T4 GPU first."
print("GPU:", torch.cuda.get_device_name(0))
ROOT = Path("/content/PolicyProof-AI")
if not ROOT.exists():
    subprocess.run(["git","clone","--depth","1",
       "https://github.com/betterlifestyle04-commits/PolicyProof-AI.git",
       str(ROOT)],check=True)
else:
    subprocess.run(["git","-C",str(ROOT),"pull","--ff-only"],check=True)
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0,str(ROOT))
print("Source commit:",subprocess.check_output(
    ["git","-C",str(ROOT),"rev-parse","HEAD"],text=True).strip())
print("NO MODEL TRAINING IN THIS CELL")


## 2. Install minimal GPU inference + native AuditKIT dependencies (no training). Avoid installing AlignTune or rerunning QLoRA.

In [ ]:
import subprocess,sys
subprocess.run([
    sys.executable,"-m","pip","install","--prefer-binary",
    "bitsandbytes==0.50.2","peft==0.21.2","auditkit==1.1.3",
],check=True)
import torch,bitsandbytes,peft,auditkit
print("GPU:",torch.cuda.get_device_name(0))
print("GPU evaluation dependencies available. NO MODEL LOAD YET.")


## 3. Upload the ORIGINAL 40-step ZIP (not a checkpoint created by repacking). Verify the exact source SHA. No model inference yet.

In [ ]:
from pathlib import Path
from google.colab import files
from scripts.dev48_gpu_pair import preflight,DEV_SHA
from scripts.verify_training_archive import digest
ARCHIVE=Path("/content/policyproof_t4_torch_extended.zip")
EXPECTED_ZIP_SHA="f76750c31e26db8603dab7f0eaef481929403ff96fd3ec6aaea1361a842fba3f"
if not ARCHIVE.is_file():
    uploaded=files.upload()
    assert len(uploaded)==1 and ARCHIVE.name in uploaded, (
       "Upload exactly the ORIGINAL policyproof_t4_torch_extended.zip.")
assert digest(ARCHIVE)==EXPECTED_ZIP_SHA, "Archive does not match the user's verified 40-step checkpoint."
print("Original checkpoint ZIP verified:",EXPECTED_ZIP_SHA)


## 4. Generate exactly the original 48-case development rows and run read-only preflight.

In [ ]:
import subprocess,sys,json
subprocess.run([sys.executable,"-m","scripts.prepare"],check=True)
from scripts.dev48_gpu_pair import preflight
assessment=preflight(ROOT,ARCHIVE,EXPECTED_ZIP_SHA)
assert assessment["status"]=="DEV48_GPU_EVALUATION_PREPARED_NOT_EXECUTED"
assert assessment["cases"]==48
print(json.dumps(assessment,indent=2,ensure_ascii=False))
print("PREFLIGHT PASSED; NO INFERENCE AND NO TRAINING PERFORMED.")


## 5. Authenticate to gated Tiny Aya without exposing the token

Use your approved Hugging Face account. If you are already logged in, this cell skips the login window. Your credentials must stay in Colab's private auth, not notebook output, GitHub or chat.

In [ ]:
from huggingface_hub import model_info,notebook_login,whoami
MODEL_ID="CohereLabs/tiny-aya-global"
MODEL_SHA="af89d219b53ed9b13b8a4645f9c8028973510324"
try:
    print("HF account:",whoami().get("name"))
    remote=model_info(MODEL_ID,revision=MODEL_SHA)
except Exception:
    notebook_login()
    remote=model_info(MODEL_ID,revision=MODEL_SHA)
assert remote.sha==MODEL_SHA
print("Approved base revision for both NEW inference passes:",remote.sha)
print("No training or model inference has occurred.")


## 6. Explicitly authorize TWO 48-case GPU inference passes (TRAINING REMAINS OFF)

**This is a fresh development comparison, not a blind test.** It loads the original 4-bit model, generates all 48 responses and saves them, frees memory, then loads the same 4-bit revision + adapter and generates 48 new responses. It uses deterministic decoding and then compares both with real AuditKIT. May use T4 quota.

Set `RUN_DEV48_GPU_EVAL=True` only after all earlier cells succeeded. On first run leave `RESUME_SAVED_BASE=False`. If the tuned pass fails **after** base JSON was saved, inspect the error; you can rerun with `RESUME_SAVED_BASE=True` in the same runtime after resolving it. Do not overwrite or delete any saved prediction files.

Training original 10-step/40-step cells MUST NOT be rerun here.

In [ ]:
RUN_DEV48_GPU_EVAL = False   # Change to True ONLY to run real inference
RESUME_SAVED_BASE = False   # Change to True ONLY if a complete base report exists
if RUN_DEV48_GPU_EVAL:
    from pathlib import Path
    from scripts.dev48_gpu_pair import execute_pair,create_results_bundle
    BASE=ROOT/"artifacts/dev48_paired_base.json"
    TUNED=ROOT/"artifacts/dev48_paired_tuned.json"
    PAIRED=ROOT/"artifacts/dev48_paired_auditkit.json"
    BUNDLE=Path("/content/policyproof_dev48_paired_gpu_results.zip")
    result=execute_pair(
        ROOT,ARCHIVE,EXPECTED_ZIP_SHA,BASE,TUNED,PAIRED,
        resume_base=RESUME_SAVED_BASE,
    )
    print("Native AuditKIT paired report status:",result["status"])
    saved=create_results_bundle(BASE,TUNED,PAIRED,BUNDLE)
    print("Evidence ZIP:",saved["archive"],saved["sha256"])
    from google.colab import files
    files.download(str(BUNDLE))
else:
    print("GPU inference OFF; model weights and optimizer were not accessed.")


## 7. CPU-only rescue if both actual prediction reports exist but AuditKIT failed

Set `RUN_CPU_AUDIT_RECOVERY=True` only if both complete **real** `BASE` and `TUNED` files exist, but no `PAIRED` file exists. This reruns only the native CPU scorer; it does not reload the model. Never substitute fabricated reports.

In [ ]:
RUN_CPU_AUDIT_RECOVERY = False
if RUN_CPU_AUDIT_RECOVERY:
    from scripts.dev48_gpu_pair import audit_existing,create_results_bundle
    from pathlib import Path
    BASE=ROOT/"artifacts/dev48_paired_base.json"
    TUNED=ROOT/"artifacts/dev48_paired_tuned.json"
    PAIRED=ROOT/"artifacts/dev48_paired_auditkit.json"
    BUNDLE=Path("/content/policyproof_dev48_paired_gpu_results.zip")
    result=audit_existing(ROOT,ARCHIVE,EXPECTED_ZIP_SHA,BASE,TUNED,PAIRED)
    print("Audit result:",result["status"])
    saved=create_results_bundle(BASE,TUNED,PAIRED,BUNDLE)
    print(saved)
    from google.colab import files
    files.download(str(BUNDLE))
else:
    print("CPU recovery OFF.")


## 8. Save results and don't call them a final benchmark

On success upload **`policyproof_dev48_paired_gpu_results.zip`** to ChatGPT, keeping the original **`policyproof_t4_torch_extended.zip`** safe. The results package contains both 48-case raw prediction reports and the AuditKIT comparison, NOT model weights. We can audit individual failures and judge whether any further training is justified.

If the runtime disconnects, download whichever complete base/tuned JSON files were saved when possible. CPU and notebook mock tests are never evidence of model improvement. A genuine final result still needs two independent reviewers on the separate frozen 60-case candidate.